In [1]:
# ! pip install -q openai datasets pandas tqdm dotenv

### Imports

In [2]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import mlflow
import pandas as pd
from utils import generate_urls, calculate_invoice_accuracies, key_level_metrics
from prompt import register_prompt
from model import log_invoice_extraction_model

from dotenv import load_dotenv
from mlflow.models import make_metric


load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

### Config

In [3]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gpt-5-nano"
REASONING = "low"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-baseline"
PROMPT_NAME = "invoice-extraction-gpt5-prompt"
PROMPT_VERSION = "1"

### Initialize MLflow and OpenAI environment

In [4]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.openai.autolog()

### Register prompt and model

In [5]:
register_prompt(prompt_name=PROMPT_NAME)

You are a Vision Language Model designed to extract structured data from invoice receipts.
    Task:
    Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

    Requirements:
    1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
    2. Preserve exact formatting for all the extracted values.  
    3. Do not output fields that lack data—omit empty keys.  
    4. Do not add any information not present in the invoice.
    5. In case of prices and currencies, ensure to maintain the original format without any modifications.

    Schema:
    {schema}

    Output:
    Return valid, minimal JSON matching this schema - no extraneous keys or null values.
    


2025/08/22 23:31:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: invoice-extraction-gpt5-prompt, version 1


In [6]:
model_info = log_invoice_extraction_model(model_name=MODEL_NAME, reasoning=REASONING, prompt_name=PROMPT_NAME, prompt_version=PROMPT_VERSION)

🏃 View run gpt-5-nano-low at: http://localhost:5000/#/experiments/438984672521325195/runs/4372c2f8e47047d08c7d67fdf12c4a16
🧪 View experiment at: http://localhost:5000/#/experiments/438984672521325195


In [7]:
mlflow.set_active_model(name=f"{MODEL_NAME}-{REASONING}")

2025/08/22 23:32:09 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-f57cab5da553469faa65253a75ce5a86


LoggedModel()

### Load the dataset

In [8]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

### Data Preparation

In [9]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

{'menu': {'nm': 'name of the menu',
  'num': 'identification number of menu',
  'unitprice': 'unit price of menu',
  'cnt': 'quantity of menu',
  'discountprice': 'discounted price of menu',
  'price': 'total price of menu',
  'itemsubtotal': 'price of each menu after discount applied',
  'vatyn': 'whether the price includes tax or not',
  'etc': 'others',
  'sub': {'nm': 'name of submenu',
   'unitprice': 'unit price of submenu',
   'cnt': 'quantity of submenu',
   'price': 'total price of submenu',
   'etc': 'others'}},
 'sub_total': {'price': 'subtotal price',
  'discount_price': 'discounted price in total',
  'service_price': 'service charge',
  'othersvc_price': 'added charge other than service charge',
  'tax_price': 'tax amount',
  'etc': 'others'},
 'total': {'total_price': 'total price',
  'etc': 'others',
  'cashprice': 'amount of price paid in cash',
  'changeprice': 'amount of change in cash',
  'creditcardprice': 'amount of price paid in credit/debit card',
  'emoneyprice'

In [10]:
test_dataset = dataset["test"].select(range(5))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

schema_dict_list = [schema_dict] * len(url_list)

eval_df = pd.DataFrame(
    {
        "schema": schema_dict_list,
        "image_base64": url_list,
    }
)

eval_df


100%|██████████| 5/5 [00:00<00:00, 12.56it/s]


,schema,image_base64
0,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


### Inference and Evaluation

In [11]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")
    
eval_with_gt_df = pd.concat([pd.DataFrame({"ground_truth": ground_truth_list}), eval_df], axis=1)
eval_with_gt_df

,ground_truth,schema,image_base64
0,"{'menu': {'nm': '-TICKET CP', 'num': '901016',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': [{'nm': 'J.STB PROMO', 'price': '1750...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'JASMINE MT ( L )', 'cnt': '1'...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'DONAT GULA', 'unitprice': '@1...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': [{'nm': 'ICE BLACKCOFFE', 'cnt': '2',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


In [12]:
def accuracy_metric(predictions, targets):
    predictions_list = [json.loads(pred) for pred in predictions]
    targets_list = targets.tolist()
    invoice_metrics_df = calculate_invoice_accuracies(targets_list, predictions_list)
    print(invoice_metrics_df)
    key_level_metrics_df = key_level_metrics(targets_list, predictions_list)
    accuracy_value = invoice_metrics_df["accuracy"].mean()
    print("Average accuracy:", accuracy_value)

    return accuracy_value

accuracy_metric = make_metric(
    eval_fn=accuracy_metric, greater_is_better=True, name="avg_accuracy"
)

In [13]:
# from mlflow.genai import scorer
# from mlflow.entities import Feedback
# import numpy as np

# eval_dataset = []
# for index, url in enumerate(url_list):
#     eval_dict = {
#         "inputs": {"image_base64": url, "schema": schema_dict},
#         "expectations": {"expected_response" : ground_truth_list[index]}
#     }
#     eval_dataset.append(eval_dict)


In [14]:
# from mlflow.genai import scorer

# @scorer
# def exact_match(inputs, outputs, expectations, trace) -> Feedback:
#     print("<<<<<<<<<Inputs: ", inputs)
#     print("<<<<<<<<<Outputs: ", outputs)
#     print("<<<<<<<<<Expectations: ", expectations)
#     print("<<<<<<<<<Trace: ", trace)
#     return Feedback(
#         value=np.random.rand(),
#         name="Accuracy"
#     )

In [15]:
# def predict_fn(image_base64, schema) -> str:
#     print("<<<<<<<<In Prediction Function - image base64: ", image_base64[:30])
#     print("<<<<<<<<In Prediction Function - schema: ", schema)
#     prompt = mlflow.genai.load_prompt(name_or_uri=PROMPT_NAME, version=PROMPT_VERSION)
#     rendered_prompt = prompt.format(schema=schema)
#     print("<<<<<<<<<<Rendered prompt: ", rendered_prompt)

#     # response = client.chat.completions.create(
#     #     model="gpt-4.1-mini", messages=rendered_prompt
#     # )
#     # return response.choices[0].message.content
#     return "dummy response"

In [16]:
# from mlflow.genai.scorers import Correctness
# results = mlflow.genai.evaluate(
#     data=eval_dataset,
#     scorers=[
#         exact_match
#     ],
#     predict_fn=predict_fn,
#     model_id=model_info.model_id
# )

In [17]:
with mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-eval") as run:
    # Perform evaluation
    results = mlflow.evaluate(
        model_info.model_uri,
        eval_with_gt_df,
        targets="ground_truth", 
        extra_metrics=[
            accuracy_metric,
        ],
    )

    # Token usage
    trace_df  = mlflow.search_traces(run_id=run.info.run_id)
    print(trace_df.shape)
    total_input_tokens = 0
    total_output_tokens = 0
    total_reasoning_tokens = 0
    for i in range(len(trace_df)):
        current_usage = trace_df["response"][i]['usage']
        total_input_tokens += current_usage['prompt_tokens']
        total_output_tokens += current_usage['completion_tokens']
        total_reasoning_tokens += current_usage['completion_tokens_details']['reasoning_tokens']

    print("Total input tokens:", total_input_tokens)
    print("Total output tokens:", total_output_tokens)
    print("Total reasoning tokens:", total_reasoning_tokens)


    # Log summary
    mlflow.log_params({
        "model_name": MODEL_NAME,
    })

    # Log invoice_metrics_df
    mlflow.log_artifact("artifacts/invoice_metrics.csv")

    mlflow.log_artifact("artifacts/key_metrics.csv")

    # log the token usage
    mlflow.log_metric("input_tokens", total_input_tokens)
    mlflow.log_metric("output_tokens", total_output_tokens)
    

    

2025/08/22 23:32:56 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-f57cab5da553469faa65253a75ce5a86
2025/08/22 23:32:56 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/08/22 23:32:57 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/08/22 23:32:57 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/08/22 23:33:17 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


   invoice_no  total_keys  matched_keys  accuracy
0           0          11             8  0.727273
False negative for key 'sub_total.subtotal_price': GT='60.000', Pred=None
False positive for key 'total.menuqty_cnt': GT='2.00', Pred='2'
False positive for key 'sub_total.price': GT=None, Pred='60.000'
False positive for key 'total.menutype_cnt': GT=None, Pred='1'
False positive for key 'menu.nm': GT='-TICKET CP', Pred='TICKET CP'
False positive for key 'menu.discountprice': GT=None, Pred='-60.000'
False positive for key 'menu.unitprice': GT=None, Pred='60.000'
Average accuracy: 0.7272727272727273
   invoice_no  total_keys  matched_keys  accuracy
0           0          11             8  0.727273
1           1           8             2  0.250000
2           2          10             8  0.800000
3           3           8             4  0.500000
4           4          14             2  0.142857
False negative for key 'sub_total.subtotal_price': GT='60.000', Pred=None
False positive for key

[Trace(trace_id=tr-c8beb40f314a1fc73c3849cfb550510c), Trace(trace_id=tr-7c59cfb79ec57d9d47a234919d9afca1), Trace(trace_id=tr-c9d19a653d4f4d52496a8033a4e264f4), Trace(trace_id=tr-38328915ea70e8cf378e57adcec29749), Trace(trace_id=tr-da746478749e1419202143a8d7f39666)]